# CRS Data Loader v2

Simple 6-phase workflow for loading CRS master data:
1. **Tenant & Branding** - Organization setup and UI customization
2. **Boundaries** - Administrative hierarchy (State → District → Block)
3. **Common Masters** - Departments, designations, complaint types
4. **Employees** - Staff accounts with roles and assignments
5. **Localizations** (Optional) - Bulk load translations for new languages
6. **Workflow** - Complaint state machine (APPLY → ASSIGN → RESOLVE)

---

In [1]:
import warnings
import pandas as pd
import requests
from openpyxl import load_workbook
from crs_loader import CRSLoader

In [2]:
# Configuration - Edit these values
URL = "http://kong:8000"
USERNAME = "ADMIN"
PASSWORD = "eGov@123"
TENANT_ID = "pg"  # Root tenant for login
TARGET_TENANT = "topping.abc"
 # Target tenant for data (can be child like "pg.citya")

loader = CRSLoader(URL)
loader.login(username=USERNAME, password=PASSWORD, tenant_id=TENANT_ID)

# Create target tenant if it doesn't exist (also enables PGR & HRMS modules)
loader.create_tenant(TARGET_TENANT, "test", users=[
        {"username": "ADMIN", "password": "eGov@123", "name": "Admin",
         "roles": ["SUPERUSER", "EMPLOYEE", "CSR", "GRO", "DGRO", "PGR_LME", "PGR_VIEWER", "CITIZEN"]}
])

loader.login(username="ADMIN", password="eGov@123", tenant_id=TENANT_ID)

✅ Authentication successful!
   User: ADMIN
   Name: System Administrator
   Tenant: pg
   Token: 135806eb-a553-42bd-b...
📝 Creating tenant 'topping.abc'...

🔧 Bootstrapping new tenant root 'topping' from 'pg'...
   📋 Schemas: 31 copied, 1 already existed, 0 failed (of 32 total)

[UPLOADING] tenant.tenants
   Tenant: topping
   Records: 1
   API URL: http://kong:8000/mdms-v2/v2/_create/tenant.tenants
   [OK] [1/1] topping
[SUMMARY] Created: 1
[SUMMARY] Already Exists: 0
[SUMMARY] Failed: 0
   ✅ Root tenant record created for 'topping'

[UPLOADING] DataSecurity.DecryptionABAC
   Tenant: topping
   Records: 25
   API URL: http://kong:8000/mdms-v2/v2/_create/DataSecurity.DecryptionABAC
   [OK] [1/25] 1
   [OK] [2/25] 2
   [OK] [3/25] 3
   [OK] [4/25] 4
   [OK] [5/25] 5
   [OK] [6/25] 6
   [OK] [7/25] 7
   [OK] [8/25] 8
   [OK] [9/25] 9
   [OK] [10/25] 10
   [OK] [11/25] 11
   [OK] [12/25] 12
   [OK] [13/25] 13
   [OK] [14/25] 14
   [OK] [15/25] 15
   [OK] [16/25] 16
   [OK] [17/25] 17
   

True

In [5]:
# Phase 2a - Generate Boundary Template
# Creates hierarchy definition and downloads an Excel template to fill

BOUNDARY_HIERARCHY = "ADMIN"  # or "ADMIN" 
BOUNDARY_LEVELS = ["State", "District", "Locality"]  # Top to bottom

template_path = loader.load_hierarchy(
    name=BOUNDARY_HIERARCHY,
    levels=BOUNDARY_LEVELS,
    target_tenant=TARGET_TENANT,
    output_dir="upload"
)
print(f"Template saved to: {template_path}")
# Fill the template with your boundary data, then run Phase 2b below


PHASE 2a: BOUNDARY HIERARCHY & TEMPLATE
Tenant: topping.abc
Hierarchy: ADMIN
Levels: State -> District -> Locality

[1/4] Building hierarchy definition...
   Created 3 level definitions

[2/4] Creating hierarchy...

⚠️  [EXISTS] Boundary hierarchy already exists
   Tenant: topping.abc
   Hierarchy Type: ADMIN
   Hierarchy already exists (OK)

[3/4] Generating template...

✅ Template generation initiated
   Task ID: 35a4c565-1ed1-4318-b995-c542c55613cb
   Status: inprogress

[4/4] Waiting for template...

⏳ Polling for template generation (max 30 attempts)...

⚠️ Template generation timed out after 30 attempts
   ERROR: Template generation failed
   Details: Unknown error
Template saved to: None


In [ ]:
# Phase 2b - Load Boundaries from filled template
# Use the template path from Phase 2a, or specify your own filled boundary file

BOUNDARY_FILE = "templates/Boundary_Master.xlsx"  # Or: "upload/boundary_filled_statea_REVENUE.xlsx"

loader.load_boundaries(BOUNDARY_FILE, target_tenant=TARGET_TENANT, hierarchy_type=BOUNDARY_HIERARCHY)

In [7]:
# Phase 3 - Common Masters
loader.load_common_masters("templates/Common and Complaint Master (7).xlsx", target_tenant=TARGET_TENANT)


PHASE 3: COMMON MASTERS
File: Common and Complaint Master (7).xlsx

[1/2] Loading departments & designations...
📥 Fetching departments from MDMS for tenant: topping.abc
   ✅ Found 2 department(s)
📥 Fetching designations from MDMS for tenant: topping.abc
   ✅ Found 2 designation(s)
📥 Fetching departments from MDMS for tenant: topping
   ✅ Found 13 department(s)
📥 Fetching designations from MDMS for tenant: topping
   ✅ Found 29 designation(s)
   Existing data on topping.abc: 2 dept(s), 2 desig(s)
   Existing data on topping: 13 dept(s), 29 desig(s)
   Departments: 2 existing (reused), 0 new (to create)
   Designations: 2 existing (reused), 0 new (to create)
📥 Fetching designations from MDMS for tenant: topping.abc
   ✅ Found 2 designation(s)
📥 Fetching departments from MDMS for tenant: topping.abc
   ✅ Found 2 department(s)
   After load: 2 dept(s), 2 desig(s) on topping.abc

[2/2] Loading complaint types...
   Creating 1 complaint types...

[UPLOADING] RAINMAKER-PGR.ServiceDefs
   Ten

{'departments': None,
 'designations': None,
 'complaint_types': {'created': 1, 'exists': 0, 'failed': 0, 'errors': []}}

In [9]:
# Phase 4 - Employees
loader.load_employees("templates/Employee_Master_Dynamic_statea.xlsx", target_tenant=TARGET_TENANT)


PHASE 4: EMPLOYEES
File: Employee_Master_Dynamic_statea.xlsx

[1/2] Reading employee data...
📥 Fetching departments from MDMS for tenant: topping.abc
   ✅ Found 2 department(s)
📥 Fetching designations from MDMS for tenant: topping.abc
   ✅ Found 2 designation(s)
📥 Fetching roles from MDMS for tenant: topping.abc
   ✅ Found 22 role(s) from MDMS
   Found 1 employees

[2/2] Creating employees...

🔐 PRE-CHECK: Validating Roles in MDMS

🔍 Checking roles in MDMS for tenant: topping.abc
   ✅ Found 22 existing roles in MDMS
   ✅ All 20 required roles already exist in MDMS

[UPLOADING] HRMS Employees
   Tenant: topping.abc
   Records: 1
   API URL: http://kong:8000/egov-hrms/employees/_create
   [OK] [1/1] COOLDUDE - Created
   [✓] [1/1] COOLDUDE - Password set via user service
[SUMMARY] Created: 1
[SUMMARY] Password Updated: 1
[SUMMARY] Already Exists: 0
[SUMMARY] Failed: 0

📝 Updating Excel file: templates/Employee_Master_Dynamic_statea.xlsx
   Sheet: Employee Master
   ✅ Status columns upda

{'created': 1, 'exists': 0, 'failed': 0, 'errors': [], 'password_updated': 1}

---

## Phase 5: Localizations (Optional)

Load bulk localization messages for a new language. Use this when you need to add Hindi, Tamil, Punjabi, or other regional languages.

**Prerequisites:**
- You need a localization Excel file with a `Localization` sheet
- Required columns: `Code`, `Message`, `Locale` (e.g., `hi_IN`, `pa_IN`)
- Optional columns: `Module`

**To add a new language:**
1. Get the localization template from `templates/localization.xlsx`
2. Fill in translations for the new locale
3. Run the cell below with `language_label` and `locale_code` to enable the language in the UI dropdown

In [ ]:
# Phase 5 - Localizations (Optional)
# For bulk translations - upload localization Excel with translated messages

# Option A: Just upload messages (no new language in dropdown)
loader.load_localizations("templates/localization.xlsx", target_tenant=TARGET_TENANT)

# Option B: Upload messages AND enable new language in UI dropdown
# Uncomment and edit the values below:

# loader.load_localizations(
#     "templates/localization.xlsx",
#     target_tenant=TARGET_TENANT,
#     language_label="ਪੰਜਾਬੀ",   # Display name in UI (e.g., "Hindi", "ਪੰਜਾਬੀ", "తెలుగు")
#     locale_code="pa_IN"         # Locale code (e.g., "hi_IN", "pa_IN", "te_IN")
# )

---

## Phase 6: Workflow Configuration

Configure the PGR complaint workflow state machine from a JSON file. This defines the complaint lifecycle:
- **APPLY** → Citizen/CSR creates complaint
- **PENDINGFORASSIGNMENT** → GRO assigns to field worker  
- **PENDINGATLME** → Field worker resolves or reassigns
- **RESOLVED/REJECTED** → Terminal states with rating option

**Template:** Use `../default-data-handler/src/main/resources/PgrWorkflowConfig.json` as a starting point.

The JSON file should have a `BusinessServices` array. Use `{tenantid}` as a placeholder - it will be replaced with your target tenant.

In [ ]:
# Phase 6 - Workflow
# Loads the PGR complaint workflow state machine from JSON file

# Default PGR workflow template (copy and modify as needed)
WORKFLOW_JSON = "templates/PgrWorkflowConfig.json"

loader.load_workflow(WORKFLOW_JSON, target_tenant=TARGET_TENANT)

---

## Rollback / Delete Data

Use these cells to undo data loading. Run only what you need to rollback.

In [ ]:
# Rollback Phase 2 - Delete Boundaries
# Deletes all boundary entities and relationships for the tenant
loader.delete_boundaries(TARGET_TENANT)

In [ ]:
# Rollback Phase 3 - Delete Common Masters
# Deletes departments, designations, and complaint types
loader.rollback_common_masters(TARGET_TENANT)

In [ ]:
# Rollback Phase 1 - Delete Tenant Config
# Deletes tenant and branding configuration
loader.rollback_tenant(TARGET_TENANT)

In [ ]:
# Delete specific MDMS schema data (granular control)
loader.delete_mdms("common-masters.Department", TARGET_TENANT)
loader.delete_mdms("common-masters.Designation", TARGET_TENANT)
loader.delete_mdms("RAINMAKER-PGR.ServiceDefs", TARGET_TENANT)

In [ ]:
# FULL RESET - Delete everything (MDMS + Boundaries)
# WARNING: This deletes ALL data for the tenant!
loader.full_reset("REVENUE", TARGET_TENANT)

**Note:** Employees (Phase 4) cannot be deleted via API - they are managed in HRMS and must be deactivated manually if needed.